# Kaggle コミットで W&B sweep agent を回すテンプレ

> **初めての人へ:** 画面のどこを押すかを含めた詳しい手順は、リポジトリの `docs/GUIDE.md` §6「Kaggle の使い方」にステップバイステップでまとめてあります。まずそちらを読んでから戻ってくるのがおすすめです。

**注意事項（読んでから実行）**

- このノートは **CPU セッション** で使う。GPU は選択しない（GPU 枠を消費しない）。
- **インターネット接続を有効化** すること（git clone・pip install・wandb 同期に必要）。Settings → Internet を On。
  - インターネット接続の有効化には **電話番号認証** が必要（Add-ons / Settings の "Get phone verified"）。
- W&B の **API キーは直書きしない**。Add-ons → Secrets に `WANDB_API_KEY` として登録し（チェックボックスを On にしてノートに紐付ける）、下のセルで読み込む。
- **編集するのは最後のセルの変数 `SWEEP_ID` と `N_RUNS` の2行だけ。** チームの sweep ID は既定値として入れてあるので、普段は `N_RUNS`（回す本数）を確認するだけでよい。
- 使い方: 右上の Save Version → 「**Save & Run All（Commit）**」で放置実行する。セルを手で1個ずつ実行する方式だと、ブラウザを閉じたとき止まる。

In [ ]:
# 1. リポジトリを clone
# 【なぜ】Kaggle が用意するマシンはまっさらで、私たちのコードが入っていない。
#         まず GitHub からコード一式をこのマシンにダウンロードする。
# 【成功すると】SB3-Team フォルダができて src/train.py などが使える状態になり、
#              %cd でそのフォルダに移動する。
# （このリポジトリは Public なのでトークン不要。ここで失敗する場合は
#   Internet 設定が Off か、電話番号認証が未了の可能性が高い）
!git clone https://github.com/shironaganegi/SB3-Team.git
%cd SB3-Team

In [ ]:
# 2. 依存ライブラリをインストール
# 【なぜ】学習コードは PyTorch・gymnasium・sb3-contrib などに依存しているが、
#         まっさらなマシンには入っていない（または版が合わない）ため。
# 【成功すると】src/train.py が動かせる環境が整う。数分かかるので気長に待つ。
!pip install -r requirements.txt

In [ ]:
# 3. W&B にログイン
# 【なぜ】学習結果を W&B のダッシュボードに送るには、このマシンが
#         「あなたのアカウントである」と名乗る必要がある。無人実行なので
#         パスワード手入力の代わりに API キーで自動ログインする。
# 【どこから読む】キーは Add-ons → Secrets に WANDB_API_KEY として登録した金庫
#         から取り出す（コードに直書きすると公開時に漏れるため）。
# 【成功すると】以降の学習結果がチームの W&B ページに自動で送られるようになる。
# （Secret not found エラーが出たら、Secrets の名前の綴りとチェックボックス On を確認）
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
!wandb login --relogin $WANDB_API_KEY

In [ ]:
# 4. sweep agent を実行（このノートの仕事の本体）
# 【なぜ】sweep は W&B 側にある「試すべきハイパラ設定を配る司令塔」。
#         wandb agent は司令塔に「次の設定をください」と問い合わせ、
#         もらった設定で src/train.py を1本学習 → 終わったらまた次をもらう、
#         を N_RUNS 回繰り返す。全員が同じ sweep ID を使えば重複なく手分けできる。
# 【成功すると】W&B の sweep ページに run が1本ずつ増えていく。
#
# 【書き換える場所】下の2つの変数だけ。他の行は触らない。
#   SWEEP_ID … 参加する sweep のID（entity/project/sweep_id 形式）。
#              チームの公式 sweep を既定値として入れてあるので普段はこのまま。
#              新しい sweep に参加するときだけ、作った人からもらったIDに貼り替える
#              （sweep は誰か1人が手元で一度だけ作る: wandb sweep sweeps/baseline_sweep.yaml）。
#   N_RUNS   … このセッションで回す本数。実測で1本 6〜10 時間かかるため、
#              セッション上限（約12時間）には1本しか収まらない。1 のままにする
#              （2以上にすると、2本目が上限到達で強制終了され Crashed になる）。

SWEEP_ID = "sai3desuyo-/bipedal-timetrial/ksgu0vds"  # チームの sweep（https://wandb.ai/sai3desuyo-/bipedal-timetrial/sweeps/ksgu0vds）
N_RUNS = 1

assert SWEEP_ID, "SWEEP_ID が空です。参加する sweep のID（例: taro/bipedal-timetrial/abc123xy）を貼ってください"
!wandb agent --count {N_RUNS} {SWEEP_ID}